<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/data_connectors/OpenDataLoaderPDFReaderDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenDataLoader PDF Reader

## Overview

[OpenDataLoader PDF](https://github.com/opendataloader-project/opendataloader-pdf) is a fast, accurate, 100% local PDF extraction engine. It runs entirely on your machine with no cloud APIs or GPU required.

[opendataloader-pdf-llamaindex](https://github.com/opendataloader-project/opendataloader-pdf-llamaindex) integrates OpenDataLoader PDF into LlamaIndex as a `BasePydanticReader`, enabling you to:
- Extract PDFs into **text, Markdown, JSON** (with bounding boxes), or **HTML**
- Get **per-page `Document` splitting** with page number metadata
- Use it directly with `SimpleDirectoryReader` via `file_extractor`
- Leverage **XY-Cut++ reading order**, table detection, AI safety filtering, and more

**Requirements:** Python 3.10+ and Java 11+ on system PATH. Java is pre-installed on Google Colab.

For the full parameter reference, see the [README](https://github.com/opendataloader-project/opendataloader-pdf-llamaindex#parameters).

## Setup

In [ ]:
%pip install -q --progress-bar off --no-warn-conflicts opendataloader-pdf-llamaindex llama-index-core llama-index-embeddings-huggingface

Verify Java is available. If this fails on Colab, run `!apt-get install -q -y default-jdk`.

In [ ]:
!java -version

Download a sample PDF to use throughout this notebook:

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

SAMPLE_DIR = Path("sample_pdfs")
SAMPLE_DIR.mkdir(exist_ok=True)

SAMPLE_PDF = SAMPLE_DIR / "attention.pdf"
if not SAMPLE_PDF.exists():
    urlretrieve("https://arxiv.org/pdf/1706.03762v7", SAMPLE_PDF)
    # Verify we got a PDF, not an error page
    with open(SAMPLE_PDF, "rb") as f:
        header = f.read(5)
    if header != b"%PDF-":
        SAMPLE_PDF.unlink()
        raise RuntimeError(
            "Downloaded file is not a valid PDF. "
            "Try downloading manually from https://arxiv.org/pdf/1706.03762v7"
        )
print(f"Sample PDF ready: {SAMPLE_PDF}")

## Basic Usage

Create a reader and load a PDF. By default, each page becomes a separate `Document` with page number metadata.

In [ ]:
from llama_index.readers.opendataloader_pdf import OpenDataLoaderPDFReader

reader = OpenDataLoaderPDFReader(format="text")
documents = reader.load_data(file_path=SAMPLE_PDF)

print(f"Number of documents: {len(documents)}")
print(f"\nPage 1 metadata: {documents[0].metadata}")
print(f"\nPage 1 text (first 500 chars):\n{documents[0].text[:500]}")

## Output Formats

OpenDataLoader PDF supports four output formats. Each serves different use cases.

> **Tip:** Set `quiet=True` to suppress the engine's CLI logging output.

### Markdown

Preserves headings, lists, and tables. Works well with `MarkdownNodeParser` for chunking.

In [ ]:
reader = OpenDataLoaderPDFReader(format="markdown", quiet=True)
docs = reader.load_data(file_path=SAMPLE_PDF)
print(docs[0].text[:500])

### JSON

Structured data with bounding boxes for every element. Useful for source citations and grounding.

In [ ]:
import json

reader = OpenDataLoaderPDFReader(format="json", quiet=True)
docs = reader.load_data(file_path=SAMPLE_PDF)

# Each page document contains valid JSON
page_data = json.loads(docs[0].text)
print(json.dumps(page_data, indent=2, ensure_ascii=False)[:800])

### HTML

Styled output preserving visual structure.

In [ ]:
reader = OpenDataLoaderPDFReader(format="html", quiet=True)
docs = reader.load_data(file_path=SAMPLE_PDF)
print(docs[0].text[:500])

## Advanced Options

### Tagged PDF Support

For accessible PDFs with structure tags (common in government/legal documents):

In [ ]:
reader = OpenDataLoaderPDFReader(
    format="markdown",
    use_struct_tree=True,
    quiet=True,
)
docs = reader.load_data(file_path=SAMPLE_PDF)
print(f"Documents: {len(docs)}")
print(docs[0].text[:300])

### Table Detection

Use `table_method="cluster"` for better detection of borderless tables:

In [ ]:
reader = OpenDataLoaderPDFReader(
    format="markdown",
    table_method="cluster",
    quiet=True,
)
docs = reader.load_data(file_path=SAMPLE_PDF)
print(docs[0].text[:500])

### Sensitive Data Sanitization

Automatically mask emails, phone numbers, IPs, credit card numbers, and URLs:

In [ ]:
reader = OpenDataLoaderPDFReader(
    format="text",
    sanitize=True,
    quiet=True,
)
docs = reader.load_data(file_path=SAMPLE_PDF)
print(docs[0].text[:300])

### Page Selection

Extract only specific pages:

In [ ]:
reader = OpenDataLoaderPDFReader(
    format="text",
    pages="1,3,5-7",
    quiet=True,
)
docs = reader.load_data(file_path=SAMPLE_PDF)
print(f"Extracted {len(docs)} pages")
for doc in docs:
    print(f"  Page {doc.metadata['page']}")

### Image Handling

Extract images embedded in PDFs:

In [ ]:
reader = OpenDataLoaderPDFReader(
    format="html",
    image_output="embedded",
    quiet=True,
)
docs = reader.load_data(file_path=SAMPLE_PDF)
print(f"Documents with embedded images: {len(docs)}")

### Other Options

Additional parameters available (see [full reference](https://github.com/opendataloader-project/opendataloader-pdf-llamaindex#parameters)):

```python
# Password-protected PDFs
reader = OpenDataLoaderPDFReader(password="secret")

# Include page headers and footers (excluded by default)
reader = OpenDataLoaderPDFReader(include_header_footer=True)

# Disable specific AI safety filters
reader = OpenDataLoaderPDFReader(content_safety_off=["hidden-text"])
```

## Custom Metadata with `extra_info`

Pass custom metadata via `extra_info` — it gets merged into every document's metadata:

In [ ]:
reader = OpenDataLoaderPDFReader(format="text", quiet=True)
docs = reader.load_data(
    file_path=SAMPLE_PDF,
    extra_info={"category": "research", "year": 2017},
)
print(docs[0].metadata)
# Custom keys are merged; reserved keys (source, format, page) are preserved

## Multiple File Loading

Pass a list of file paths to load multiple PDFs in one call:

In [ ]:
# Copy sample to create a second file for demonstration
import shutil

SAMPLE_PDF_2 = SAMPLE_DIR / "attention_copy.pdf"
shutil.copy2(SAMPLE_PDF, SAMPLE_PDF_2)

reader = OpenDataLoaderPDFReader(format="text", quiet=True)
docs = reader.load_data(file_path=[SAMPLE_PDF, SAMPLE_PDF_2])
print(f"Total documents from 2 files: {len(docs)}")

# Each document's source metadata identifies the originating file
sources = {doc.metadata["source"] for doc in docs}
print(f"Sources: {sources}")

SAMPLE_PDF_2.unlink()  # Clean up copy

## Document Metadata

Each `Document` includes metadata describing its origin:

| Key | Present When | Description |
|-----|-------------|-------------|
| `source` | Always | PDF filename |
| `format` | Always | Output format (`text`, `markdown`, `json`, `html`) |
| `page` | `split_pages=True` (default) | Page number (1-indexed) |
| `hybrid` | Hybrid mode enabled | Backend name (e.g., `docling-fast`) |

When using `SimpleDirectoryReader`, additional file metadata (path, size, dates) is merged via `extra_info`.

## SimpleDirectoryReader Integration

Use `OpenDataLoaderPDFReader` as a `file_extractor` with LlamaIndex's `SimpleDirectoryReader`:

In [ ]:
from llama_index.core import SimpleDirectoryReader

dir_reader = SimpleDirectoryReader(
    input_dir=SAMPLE_DIR,
    file_extractor={".pdf": OpenDataLoaderPDFReader(format="markdown", quiet=True)},
)
documents = dir_reader.load_data()

print(f"Loaded {len(documents)} documents from directory")
for doc in documents[:3]:
    print(f"  Page {doc.metadata.get('page', '?')}: {doc.text[:80]}...")

## Hybrid AI Mode

For higher accuracy on complex documents, OpenDataLoader PDF can use a hybrid AI backend (`docling-fast`). This requires a running backend server.

```python
# Requires a running hybrid backend server
reader = OpenDataLoaderPDFReader(
    format="markdown",
    hybrid="docling-fast",
    hybrid_fallback=True,  # Fall back to Java engine on backend failure
)
docs = reader.load_data(file_path="document.pdf")
```

When hybrid mode is active, document metadata includes the `hybrid` key:
```python
{"source": "document.pdf", "format": "markdown", "page": 1, "hybrid": "docling-fast"}
```

## Whole-Document Mode

By default, `split_pages=True` creates one `Document` per page. Set `split_pages=False` to get one `Document` per file:

In [ ]:
reader = OpenDataLoaderPDFReader(format="text", split_pages=False, quiet=True)
docs = reader.load_data(file_path=SAMPLE_PDF)

print(f"Documents: {len(docs)} (whole file as single document)")
print(f"Metadata: {docs[0].metadata}")
print(f"Text length: {len(docs[0].text)} chars")

## RAG Pipeline Example

Build a complete RAG pipeline with OpenDataLoader PDF and LlamaIndex.

This example uses a local embedding model to build a vector index and retrieve relevant chunks. Optionally, uncomment the LLM block at the end to run a full query (requires `HF_TOKEN`).

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.readers.opendataloader_pdf import OpenDataLoaderPDFReader

# Load PDFs with Markdown format (preserves structure for better chunking)
dir_reader = SimpleDirectoryReader(
    input_dir=SAMPLE_DIR,
    file_extractor={".pdf": OpenDataLoaderPDFReader(format="markdown", quiet=True)},
)
documents = dir_reader.load_data()
print(f"Loaded {len(documents)} documents")

# Build index with MarkdownNodeParser and a local embedding model (~130MB download)
node_parser = MarkdownNodeParser()
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
index = VectorStoreIndex.from_documents(
    documents, transformations=[node_parser], embed_model=embed_model
)

# Retrieve relevant chunks (no LLM needed)
QUERY = "What is the Transformer architecture?"
retriever = index.as_retriever(similarity_top_k=3)
nodes = retriever.retrieve(QUERY)
print(f"Q: {QUERY}\n\nTop {len(nodes)} retrieved chunks:")
for i, node in enumerate(nodes, 1):
    print(f"\n--- Chunk {i} (score: {node.score:.4f}) ---")
    print(node.text[:200])
    print(f"Metadata: {node.metadata}")

# Optional: full RAG query with an LLM (requires HF_TOKEN in Colab secrets)
# %pip install -q --progress-bar off llama-index-llms-huggingface-api
# import os
# from google.colab import userdata  # Colab only; remove if running locally
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # bridge secret to env var
# from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
# llm = HuggingFaceInferenceAPI(
#     token=os.getenv("HF_TOKEN"),
#     model_name="mistralai/Mixtral-8x7B-Instruct-v0.1",
# )
# response = index.as_query_engine(llm=llm).query(QUERY)
# print(f"A: {response.response.strip()}")

## Cleanup

In [ ]:
import shutil

shutil.rmtree(SAMPLE_DIR, ignore_errors=True)
print("Cleaned up sample files.")